# Kiểm lại phân tích do LLM viết, và chuẩn bị dữ liệu an toàn trước khi hỏi AI

Một đồng nghiệp dán mô tả bộ `bike-sharing-uci` vào một chatbot và nhận về tám "insight" kèm con số, định gửi
thẳng cho giám đốc vận hành. Trước khi gửi, bạn kiểm từng con số bằng code của mình. Sau đó, vì tuần sau team muốn dùng
AI trên dữ liệu đơn hàng có mã khách, bạn viết một hàm tạo **hồ sơ bảng an toàn để gửi LLM**: đủ để LLM viết được SQL,
không lộ dữ liệu từng dòng hay mã định danh.

Dữ liệu: `day.csv` (731 ngày) và `hour.csv` (17 379 giờ), Capital Bikeshare 2011-2012. Data dictionary (tệp
`Readme.txt` của UCI): `weekday` 0 = Chủ nhật ... 6 = Thứ Bảy; `season` 1-4 (nhãn trong Readme: 1 spring, 2 summer, 3 fall,
4 winter; xem lưu ý ở module 08); `temp` là nhiệt độ **chuẩn hoá**, chia cho 41 (°C = `temp * 41`); `casual` khách vãng
lai, `registered` thành viên, `cnt = casual + registered`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Cell gắn tag `explore` bị bỏ qua khi chấm.
- Viết `actual_values`, `verify` (phần 1-2) và `llm_safe_profile` (phần 3).
- Chạy test: `uv run pytest tracks/data-analyst/modules/15-stakeholder-storytelling-interview/exercises/intermediate/02-ai-output-verification -q`.
- Xong thì làm phần **trắc nghiệm** trong app.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc day.csv và hour.csv
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

BIKE = ensure("bike-sharing-uci")
day = pd.read_csv(BIKE / "day.csv", parse_dates=["dteday"])
hour = pd.read_csv(BIKE / "hour.csv", parse_dates=["dteday"])
print(day.shape, hour.shape)

In [ ]:
import re

import numpy as np
import pandas as pd

# Tám "insight" chatbot trả về, chép nguyên văn. claimed là con số trong câu, theo đơn vị của khoá tương ứng
# trong actual_values (tỉ lệ 0-1, lượt, °C, mã weekday theo data dictionary: Thứ Bảy = 6).
LLM_CLAIMS = pd.DataFrame([
    ("yoy_growth", "Tổng lượt thuê năm 2012 tăng 64.9% so với năm 2011.", 0.649),
    ("best_weekday", "Thứ Bảy là ngày đông nhất trong tuần, trung bình 4 690 lượt/ngày.", 6),
    ("summer_mean", "Mùa hè (season = 2) đông nhất, trung bình 5 644 lượt/ngày.", 5644),
    ("mean_temp_c", "Nhiệt độ trung bình hai năm là 0.50 độ C.", 0.50),
    ("casual_growth", "Lượt của khách vãng lai năm 2012 tăng 68.4% so với 2011.", 0.684),
    ("peak_hour_mean", "Giờ cao điểm là 17h với trung bình 461.5 lượt/giờ.", 461.5),
    ("peak_day_count", "Ngày đông nhất là 15/9/2012 với 8 714 lượt.", 8714),
    ("temp_corr", "Tương quan giữa nhiệt độ và lượt thuê theo ngày là 0.63.", 0.63),
], columns=["claim_id", "text", "claimed"])

SENSITIVE_TOKENS = {"id", "email", "phone", "name", "address", "dob"}
REDACTED = "<redacted>"

## 1. Tính lại từng con số

`actual_values(day, hour) -> dict` trả **đúng tám khoá** trùng `claim_id`, giá trị là `float`, tính từ dữ liệu theo đúng
data dictionary:

| Khoá | Cách tính |
| --- | --- |
| `yoy_growth` | Tổng `cnt` năm 2012 / tổng năm 2011 − 1 (năm lấy từ `dteday`) |
| `best_weekday` | Mã `weekday` có `cnt` trung bình mỗi ngày cao nhất (trong `day`) |
| `summer_mean` | `cnt` trung bình mỗi ngày của `season == 2` |
| `mean_temp_c` | Trung bình `temp * 41` trên `day` |
| `casual_growth` | Tổng `casual` 2012 / tổng 2011 − 1 |
| `peak_hour_mean` | Lớn nhất trong các trung bình `cnt` theo `hr` (trong `hour`) |
| `peak_day_count` | `cnt` lớn nhất của một ngày |
| `temp_corr` | Hệ số tương quan Pearson giữa `temp` và `cnt` trong `day` |

`dteday` có thể là chuỗi hoặc datetime. Không sửa đầu vào.

In [ ]:
def actual_values(day: pd.DataFrame, hour: pd.DataFrame) -> dict:
    """Giá trị thật cho tám claim_id (xem bảng ở trên)."""
    # TODO: year = pd.to_datetime(day["dteday"]).dt.year
    # TODO: tám phép tính, mỗi giá trị ép float
    raise NotImplementedError

## 2. Đối chiếu claim với số thật

`verify(claims, actual, rel_tol=0.01) -> DataFrame` nhận bảng có cột `claim_id`, `claimed` (các cột khác bỏ qua) và dict
`actual`, trả một dòng mỗi claim **theo đúng thứ tự** đầu vào, index 0..n-1, cột `claim_id`, `claimed`, `actual`,
`rel_error`, `ok`:

- `rel_error = |claimed − actual| / |actual|`; nếu `actual == 0` thì dùng `|claimed − actual|`.
- `ok = rel_error <= rel_tol` (kiểu `bool`).
- Claim không có trong `actual`: `actual` và `rel_error` là `NaN`, `ok = False`. Con số không kiểm được thì chưa được gửi.

In [ ]:
def verify(claims: pd.DataFrame, actual: dict, rel_tol: float = 0.01) -> pd.DataFrame:
    """Bảng đối chiếu claim với giá trị thật."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kết quả đối chiếu trên dữ liệu thật
report = verify(LLM_CLAIMS, actual_values(day, hour))
report.assign(text=LLM_CLAIMS["text"]).round(4)

In [ ]:
# explore: ba claim sai vì hiểu sai data dictionary - chatbot đã "nghĩ" gì?
print("TB lượt/ngày theo weekday (0 = Chủ nhật):", day.groupby("weekday")["cnt"].mean().round(0).to_dict())
print("TB lượt/ngày theo season:", day.groupby("season")["cnt"].mean().round(0).to_dict())
print("temp TB (chuẩn hoá):", round(day["temp"].mean(), 3), "-> °C:", round((day["temp"] * 41).mean(), 1))

## 3. Hồ sơ bảng an toàn để gửi LLM

Để nhờ LLM viết SQL, nó cần **tên cột, kiểu, mức thiếu, số giá trị khác nhau và một ví dụ**, không cần dữ liệu từng dòng.
`llm_safe_profile(df, sensitive=()) -> DataFrame`:

- Một dòng mỗi cột của `df`, index là tên cột theo thứ tự gốc; cột `dtype` (chuỗi, ví dụ `"int64"`), `missing` (số ô
  thiếu, `int`), `unique` (số giá trị khác nhau, không tính thiếu, `int`), `example` (chuỗi).
- Cột **nhạy cảm** là cột trong `sensitive` **hoặc** có tên chứa một token trong `SENSITIVE_TOKENS` (tách tên cột viết
  thường theo ký tự không phải chữ/số: `"Customer ID"` → `customer`, `id`; `"e-mail"` không khớp, `"email"` khớp).
  `example` của cột nhạy cảm là `REDACTED`; cột khác là `str` của giá trị khác thiếu đầu tiên (cột toàn thiếu: `""`).
- Tên trong `sensitive` không có trong `df` → `ValueError` (gõ sai tên cột nhạy cảm là lỗi nguy hiểm, không được lờ đi).
- Không sửa `df`.

In [ ]:
def llm_safe_profile(df: pd.DataFrame, sensitive=()) -> pd.DataFrame:
    """Hồ sơ cột (dtype, missing, unique, example) với ví dụ của cột nhạy cảm bị che."""
    # TODO: kiểm sensitive có trong df, nếu không raise ValueError
    # TODO: với mỗi cột: tokens = re.split(r"[^0-9a-z]+", tên.lower()); nhạy cảm nếu trong sensitive hoặc giao SENSITIVE_TOKENS
    raise NotImplementedError

In [ ]:
# explore: hồ sơ của day.csv - đây là thứ có thể dán vào chatbot kèm câu hỏi, thay cho dữ liệu
print(llm_safe_profile(day).to_string())

## 4. Trước khi gửi giám đốc

1. Bốn claim sai có chung một nguyên nhân gốc nào? (Gợi ý: so con số chatbot đưa ra với các bảng ở cell explore phần 2.)
2. Claim `temp_corr` đúng về số. Nếu chatbot viết tiếp "vậy tăng nhiệt độ sẽ tăng lượt thuê", câu đó sai ở đâu? (Module
   08, phần 4: tương quan trong mùa hè.)
3. Claim `peak_hour_mean` đúng, nhưng đó là trung bình của mọi ngày. Câu đó có đúng cho cuối tuần không?
4. Hồ sơ an toàn có đủ để LLM viết câu SQL "lượt thuê trung bình theo thứ" không? Nó cần thêm gì (data dictionary)?

Xong thì mở app, làm phần **trắc nghiệm** của bài.